# DQX Profiling: Food Inspections (Chicago & Dallas)
##### Varun Singh | 002301584

Install DQX library

In [0]:
%pip install databricks-labs-dqx

In [0]:
dbutils.library.restartPython()

Re-read parameters after restart, load Bronze tables

In [0]:
from databricks.labs.dqx.profiler.profiler import DQProfiler
from databricks.sdk import WorkspaceClient
from pyspark.sql.functions import col, count, when, lit, round as spark_round, length, substring
import pandas as pd

CATALOG = dbutils.widgets.get("catalog")
BRONZE  = dbutils.widgets.get("bronze")

chicago_df = spark.table(f"{CATALOG}.{BRONZE}.chicago_inspections")
dallas_df  = spark.table(f"{CATALOG}.{BRONZE}.dallas_inspections")

print(f"Chicago: {chicago_df.count()} rows, {len(chicago_df.columns)} columns")
print(f"Dallas: {dallas_df.count()} rows, {len(dallas_df.columns)} columns")

Built-in Databricks profiler for Chicago

In [0]:
dbutils.data.summarize(chicago_df)

Built-in Databricks profiler for Dallas

In [0]:
dbutils.data.summarize(dallas_df)

DQX profiler on Chicago. Note: DQX internally samples data for profiling statistics. Full dataset analysis is done in the null/distribution cells below.

In [0]:
w = WorkspaceClient()
profiler = DQProfiler(w)

chicago_stats, chicago_rules = profiler.profile(chicago_df)

# column stats
print("=" * 90)
print("CHICAGO DQX COLUMN STATISTICS")
print("=" * 90)

stats_rows = []
for col_name, stats in chicago_stats.items():
    stats_rows.append({
        "column": col_name,
        "total": stats.get("count", ""),
        "non_null": stats.get("count_non_null", ""),
        "nulls": stats.get("count_null", ""),
        "null_pct": f"{round(stats.get('count_null', 0) / stats.get('count', 1) * 100, 1)}%",
        "min": str(stats.get("min", ""))[:30],
        "max": str(stats.get("max", ""))[:30],
        "mean": str(stats.get("mean", ""))[:15]
    })

chicago_stats_pdf = pd.DataFrame(stats_rows)
pd.set_option("display.max_colwidth", 35)
pd.set_option("display.width", 200)
print(chicago_stats_pdf.to_string(index=False))

# generated rules
print("\n" + "=" * 90)
print("CHICAGO DQX GENERATED RULES")
print("=" * 90)
print(f"{'Column':<30} {'Rule':<25} {'Description'}")
print("~" * 90)
for rule in chicago_rules:
    desc = str(rule.description)[:60] if rule.description else ""
    print(f"{rule.column:<30} {rule.name:<25} {desc}")

DQX profiler on Dallas

In [0]:
dallas_stats, dallas_rules = profiler.profile(dallas_df)

print("=" * 90)
print("DALLAS DQX COLUMN STATISTICS")
print("=" * 90)

stats_rows = []
for col_name, stats in dallas_stats.items():
    stats_rows.append({
        "column": col_name[:35],
        "total": stats.get("count", ""),
        "non_null": stats.get("count_non_null", ""),
        "nulls": stats.get("count_null", ""),
        "null_pct": f"{round(stats.get('count_null', 0) / stats.get('count', 1) * 100, 1)}%",
        "min": str(stats.get("min", ""))[:30],
        "max": str(stats.get("max", ""))[:30],
        "mean": str(stats.get("mean", ""))[:15]
    })

dallas_stats_pdf = pd.DataFrame(stats_rows)
print(dallas_stats_pdf.to_string(index=False))

print("\n" + "=" * 90)
print("DALLAS DQX GENERATED RULES")
print("=" * 90)
print(f"{'Column':<35} {'Rule':<25} {'Description'}")
print("~" * 90)
for rule in dallas_rules:
    desc = str(rule.description)[:55] if rule.description else ""
    print(f"{rule.column[:35]:<35} {rule.name:<25} {desc}")

Chicago null analysis on the full dataset (308,431 rows)

In [0]:
print("=" * 90)
print("CHICAGO NULL ANALYSIS (FULL DATASET)")
print("=" * 90)

chicago_total = chicago_df.count()

chicago_null_counts = chicago_df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in chicago_df.columns
])

chicago_nulls = chicago_null_counts.toPandas().T
chicago_nulls.columns = ["null_count"]
chicago_nulls["null_pct"] = (chicago_nulls["null_count"] / chicago_total * 100).round(2)
chicago_nulls["non_null"] = chicago_total - chicago_nulls["null_count"]
chicago_nulls = chicago_nulls.sort_values("null_pct", ascending=False)

print(f"Total rows: {chicago_total}\n")
print(f"{'Column':<25} {'Nulls':>10} {'Non Null':>10} {'Null %':>10}")
print("~" * 60)
for idx, row in chicago_nulls.iterrows():
    print(f"{idx:<25} {int(row['null_count']):>10} {int(row['non_null']):>10} {row['null_pct']:>9}%")

Dallas null analysis on the full dataset (78,984 rows)

In [0]:
print("=" * 90)
print("DALLAS NULL ANALYSIS (FULL DATASET)")
print("=" * 90)

dallas_total = dallas_df.count()

dallas_core_cols = [
    "Restaurant Name", "Inspection Type", "Inspection Date", "Inspection Score",
    "Street Number", "Street Name", "Street Direction", "Street Type", "Street Unit",
    "Street Address", "Zip Code", "Inspection Month", "Inspection Year", "Lat Long Location"
]
violation_cols = [f"Violation Description - {i}" for i in range(1, 11)]
dallas_analyze_cols = dallas_core_cols + violation_cols + ["_source_file", "_source_city", "_load_timestamp"]

dallas_null_counts = dallas_df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in dallas_analyze_cols
])

dallas_nulls = dallas_null_counts.toPandas().T
dallas_nulls.columns = ["null_count"]
dallas_nulls["null_pct"] = (dallas_nulls["null_count"] / dallas_total * 100).round(2)
dallas_nulls["non_null"] = dallas_total - dallas_nulls["null_count"]
dallas_nulls = dallas_nulls.sort_values("null_pct", ascending=False)

print(f"Total rows: {dallas_total}\n")
print(f"{'Column':<30} {'Nulls':>10} {'Non Null':>10} {'Null %':>10}")
print("~" * 65)
for idx, row in dallas_nulls.iterrows():
    print(f"{idx:<30} {int(row['null_count']):>10} {int(row['non_null']):>10} {row['null_pct']:>9}%")

Chicago column value distributions: Results, Inspection Type, Risk, Facility Type, Zip, City, State

In [0]:
print("=" * 90)
print("CHICAGO: KEY COLUMN DISTRIBUTIONS")
print("=" * 90)

print("\n--- Results Distribution ---")
chicago_df.groupBy("Results").count().orderBy(col("count").desc()).show(truncate=False)

print("--- Inspection Type Distribution ---")
chicago_df.groupBy("Inspection Type").count().orderBy(col("count").desc()).show(20, truncate=False)

print("--- Risk Distribution ---")
chicago_df.groupBy("Risk").count().orderBy(col("count").desc()).show(truncate=False)

print("--- Facility Type (Top 15) ---")
chicago_df.groupBy("Facility Type").count().orderBy(col("count").desc()).show(15, truncate=False)

print("--- Top 10 Zip Codes ---")
chicago_df.groupBy("Zip").count().orderBy(col("count").desc()).show(10, truncate=False)

print("--- City Values (check for inconsistencies) ---")
chicago_df.groupBy("City").count().orderBy(col("count").desc()).show(truncate=False)

print("--- State Values ---")
chicago_df.groupBy("State").count().orderBy(col("count").desc()).show(truncate=False)

Dallas column value distributions: Inspection Type, Score, Zip, Restaurants, Street Direction

In [0]:
print("=" * 90)
print("DALLAS: KEY COLUMN DISTRIBUTIONS")
print("=" * 90)

print("\n--- Inspection Type Distribution ---")
dallas_df.groupBy("Inspection Type").count().orderBy(col("count").desc()).show(truncate=False)

print("--- Inspection Score Distribution ---")
dallas_df.groupBy("Inspection Score").count().orderBy(col("Inspection Score").desc()).show(20, truncate=False)

print("--- Score Statistics ---")
dallas_df.select("Inspection Score").describe().show()

print("--- Scores Above 100 (Invalid per assignment rules) ---")
invalid_scores = dallas_df.filter(col("Inspection Score") > 100).count()
print(f"Rows with score > 100: {invalid_scores}")

print("\n--- Top 10 Zip Codes ---")
dallas_df.groupBy("Zip Code").count().orderBy(col("count").desc()).show(10, truncate=False)

print("--- Top 10 Restaurants by Inspection Count ---")
dallas_df.groupBy("Restaurant Name").count().orderBy(col("count").desc()).show(10, truncate=False)

print("--- Street Direction Values ---")
dallas_df.groupBy("Street Direction").count().orderBy(col("count").desc()).show(truncate=False)

Chicago violations column: null rate, sample strings, length stats

In [0]:
print("=" * 90)
print("CHICAGO: VIOLATIONS COLUMN ANALYSIS")
print("=" * 90)

violations_null = chicago_df.filter(col("Violations").isNull()).count()
violations_not_null = chicago_df.filter(col("Violations").isNotNull()).count()
print(f"Null violations: {violations_null} ({round(violations_null/chicago_total*100, 2)}%)")
print(f"Non null violations: {violations_not_null} ({round(violations_not_null/chicago_total*100, 2)}%)")

print("\n--- Sample Violations (first 3 non null, truncated) ---")
chicago_df.filter(col("Violations").isNotNull()) \
    .select("Inspection ID", "Results", substring("Violations", 1, 300).alias("Violations_Preview")) \
    .show(3, truncate=False)

print("--- Violations String Length Stats ---")
chicago_df.filter(col("Violations").isNotNull()) \
    .select(length("Violations").alias("violation_length")) \
    .describe().show()

Dallas violation slot sparsity: how many rows have data in each of the 25 slots

In [0]:
print("=" * 90)
print("DALLAS: VIOLATION COLUMNS SPARSITY ANALYSIS")
print("=" * 90)
print("Shows how many rows have data in each violation slot\n")

violation_stats = []
for i in range(1, 26):
    desc_col = f"Violation Description - {i}"
    if desc_col in dallas_df.columns:
        filled = dallas_df.filter(col(desc_col).isNotNull()).count()
        pct = round(filled / dallas_total * 100, 2)
        violation_stats.append({
            "slot": i, 
            "filled_rows": filled, 
            "null_rows": dallas_total - filled,
            "filled_pct": f"{pct}%"
        })

violation_sparsity = pd.DataFrame(violation_stats)
print(violation_sparsity.to_string(index=False))

max_used_slot = violation_sparsity[violation_sparsity["filled_rows"] > 0]["slot"].max()
print(f"\nMax violation slot with any data: {max_used_slot}")
print(f"Recommendation: Unpivot slots 1 through {max_used_slot} in Silver layer")

Schema comparison: shared columns, type mismatches, columns unique to each city

In [0]:
# compare columns across both datasets

chi_cols = set(c for c in chicago_df.columns if not c.startswith("_"))
dal_cols = set(c for c in dallas_df.columns if not c.startswith("_"))

shared = chi_cols & dal_cols
chi_only = chi_cols - dal_cols
dal_only = dal_cols - chi_cols

print("=" * 90)
print("SCHEMA COMPARISON")
print("=" * 90)

print(f"\nChicago: {len(chi_cols)} columns | Dallas: {len(dal_cols)} columns")

print(f"\nShared column names ({len(shared)}):")
for c in sorted(shared):
    chi_type = str(dict((f.name, f.dataType) for f in chicago_df.schema.fields)[c])
    dal_type = str(dict((f.name, f.dataType) for f in dallas_df.schema.fields)[c])
    match_flag = "OK" if chi_type == dal_type else f"TYPE MISMATCH: {chi_type} vs {dal_type}"
    print(f"  {c:<30} {match_flag}")

print(f"\nChicago only ({len(chi_only)}):")
for c in sorted(chi_only):
    print(f"  {c}")

print(f"\nDallas only ({len(dal_only)}):")
dal_viol = sorted([c for c in dal_only if "Violation" in c])
dal_other = sorted([c for c in dal_only if "Violation" not in c])
for c in dal_other:
    print(f"  {c}")
print(f"  + {len(dal_viol)} violation slot columns (Description/Points/Detail/Memo x 25)")

print("\nManual mappings needed (different names, same concept):")
for chi, dal in [("DBA Name","Restaurant Name"),("Zip","Zip Code"),("Address","Street Address"),("Location","Lat Long Location"),("Results","Inspection Score")]:
    print(f"  {chi:<25} <=> {dal}")

Data quality findings: casing issues, score ranges, null rates, column name typos

In [0]:
print("=" * 90)
print("DATA QUALITY FINDINGS")
print("=" * 90)

# chicago city casing
print("\n[Chicago] City column distinct values:")
chicago_df.groupBy("City").count().orderBy(col("count").desc()).show(10, truncate=False)

# chicago violations null rate
chi_viol_null = chicago_df.filter(col("Violations").isNull()).count()
chi_viol_filled = chicago_total - chi_viol_null
print(f"[Chicago] Violations: {chi_viol_filled:,} filled, {chi_viol_null:,} null ({round(chi_viol_null/chicago_total*100,1)}%)")

# sample violation string
print("\n[Chicago] Sample violation string (first 200 chars):")
sample = chicago_df.filter(col("Violations").isNotNull()).select("Violations").first()[0]
print(f"  {sample[:200]}...")
print(f"  Separator: ' | ', pattern: 'code. DESCRIPTION ... Comments: ...'")

# dallas score range
print("\n[Dallas] Inspection Score range:")
dallas_df.select("Inspection Score").summary("min", "max", "mean").show()

inv = dallas_df.filter(col("Inspection Score") > 100).count()
neg = dallas_df.filter(col("Inspection Score") < 0).count()
print(f"  Scores > 100: {inv} | Scores < 0: {neg}")

# dallas sparsity quick check
print("\n[Dallas] Violation slot fill rates (slots 1, 5, 10, 15, 20, 25):")
for slot in [1, 5, 10, 15, 20, 25]:
    c = f"Violation Description - {slot}"
    if c in dallas_df.columns:
        filled = dallas_df.filter(col(c).isNotNull()).count()
        print(f"  Slot {slot:>2}: {filled:>6} filled ({round(filled/dallas_total*100,1)}%)")

# dallas column name typo
print("\n[Dallas] Column names with double spaces (typos):")
typo_cols = [c for c in dallas_df.columns if "  " in c]
for c in typo_cols:
    print(f"  '{c}'")
if not typo_cols:
    print("  None found")

# dallas lat long nulls
dal_loc_null = dallas_df.filter(col("Lat Long Location").isNull()).count()
print(f"\n[Dallas] Lat Long Location nulls: {dal_loc_null:,} ({round(dal_loc_null/dallas_total*100,1)}%)")

dal_dir_null = dallas_df.filter(col("Street Direction").isNull()).count()
print(f"[Dallas] Street Direction nulls: {dal_dir_null:,} ({round(dal_dir_null/dallas_total*100,1)}%)")

Chicago column data types and max string widths

In [0]:
print("=" * 90)
print("CHICAGO: DATA TYPES AND MAX WIDTHS")
print("=" * 90)

rows = []
for f in chicago_df.schema.fields:
    if f.name.startswith("_"):
        continue
    dtype = str(f.dataType).replace("()", "")
    mx = ""
    if dtype == "StringType":
        mx = chicago_df.filter(col(f.name).isNotNull()) \
            .selectExpr(f"MAX(LENGTH(`{f.name}`))").collect()[0][0]
    rows.append((f.name, dtype, f.nullable, mx))

print(f"{'Column':<25} {'Type':<15} {'Nullable':<10} {'Max Len':<10}")
print("~" * 60)
for name, dtype, nullable, mx in rows:
    print(f"{name:<25} {dtype:<15} {str(nullable):<10} {str(mx):<10}")

Dallas column data types and max string widths (core columns only)

In [0]:
print("=" * 90)
print("DALLAS: DATA TYPES AND MAX WIDTHS (CORE COLUMNS)")
print("=" * 90)

check_cols = [
    "Restaurant Name", "Inspection Type", "Inspection Date", "Inspection Score",
    "Street Number", "Street Name", "Street Direction", "Street Type", "Street Unit",
    "Street Address", "Zip Code", "Inspection Month", "Inspection Year", "Lat Long Location",
    "Violation Description - 1", "Violation Points - 1", "Violation Detail - 1", "Violation Memo - 1"
]

rows = []
for f in dallas_df.schema.fields:
    if f.name.startswith("_") or f.name not in check_cols:
        continue
    dtype = str(f.dataType).replace("()", "")
    mx = ""
    if dtype == "StringType":
        mx = dallas_df.filter(col(f.name).isNotNull()) \
            .selectExpr(f"MAX(LENGTH(`{f.name}`))").collect()[0][0]
    rows.append((f.name, dtype, f.nullable, mx))

print(f"{'Column':<30} {'Type':<15} {'Nullable':<10} {'Max Len':<10}")
print("~" * 65)
for name, dtype, nullable, mx in rows:
    print(f"{name:<30} {dtype:<15} {str(nullable):<10} {str(mx):<10}")

Profiling summary

In [0]:
# final numbers

violations_null = chicago_df.filter(col("Violations").isNull()).count()
invalid_scores = dallas_df.filter(col("Inspection Score") > 100).count()

chi_inspections = chicago_df.select("Inspection ID").distinct().count()
chi_businesses = chicago_df.select("DBA Name").distinct().count()
chi_dates = chicago_df.selectExpr("MIN(`Inspection Date`)", "MAX(`Inspection Date`)").collect()[0]

dal_restaurants = dallas_df.select("Restaurant Name").distinct().count()
dal_dates = dallas_df.selectExpr("MIN(`Inspection Date`)", "MAX(`Inspection Date`)").collect()[0]
dal_avg = dallas_df.selectExpr("ROUND(AVG(`Inspection Score`), 2)").collect()[0][0]

print("=" * 90)
print("PROFILING SUMMARY")
print("=" * 90)

print(f"""
CHICAGO
  {chicago_total:,} rows, {len(chicago_df.columns)} cols
  {chi_inspections:,} unique inspections, {chi_businesses:,} unique businesses
  dates: {chi_dates[0]} to {chi_dates[1]}
  violations null: {violations_null:,} ({round(violations_null/chicago_total*100, 2)}%)

DALLAS
  {dallas_total:,} rows, {len(dallas_df.columns)} cols
  {dal_restaurants:,} unique restaurants
  dates: {dal_dates[0]} to {dal_dates[1]}
  avg score: {dal_avg}, scores > 100: {invalid_scores}

SILVER LAYER TODO
  1. parse chicago violations string into code/description/comment
  2. unpivot dallas 25 violation slots to long format
  3. derive chicago scores from Results text
  4. fix city/state casing inconsistencies
  5. standardize zip formats (chicago int vs dallas string)
  6. deduplicate violations per inspection
  7. apply assignment validation rules (nulls, score caps, etc.)
""")